In [3]:
import pandas as pd 
from langchain_community.llms import Ollama
from langchain_classic.agents import AgentExecutor, create_structured_chat_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.tools import tool

In [8]:
#ici il nous faut une fonction qui transforme notre excel en data frame pandas

def lire_planning_excel(chemin_fichier: str) -> pd.DataFrame :
    """ Utile pour lire le contenu du fichier Excel """
    return pd.read_excel(chemin_fichier)
lire_planning_excel("data/planning.xlsx")

,Identifiant,Qualification : Connaissance de ligne,Heure de début,Heure de fin,Type de journée,Code journée,12/01/2026
0,1,"66, 74, 137, 138, 140, 166, 174, 177, 178, 235...",NaN,NaN,NaN,NaN,NaN
1,2,"66, 74, 85, 140, 166, 175, 235, 238, 340, 363",NaN,NaN,NaN,NaN,NaN
2,3,"140, 175, 235, 238, 340, 538, 860",00:00,00:00 (+1),NaN,R,R
3,4,"66, 74, 85, 137, 138, 140, 174, 175, 177, 235,...",NaN,NaN,NaN,NaN,NaN
4,5,"66, 74, 85, 137, 138, 166, 174, 177, 178, 237,...",00:00,00:00 (+1),NaN,R,R
...,...,...,...,...,...,...,...
880,881,NaN,12:00,19:30,JOUR,L175002,L175002
881,882,NaN,12:00,19:30,JOUR,L175002,L175002
882,883,NaN,12:00,19:30,JOUR,L175002,L175002
883,884,NaN,12:00,19:30,JOUR,L175002,L175002


In [14]:
# Configuration d'Ollama et de l'Agent IA

llm = Ollama(model="llama3", temperature=0)

# Liste des outils mis à disposition de l'agent
tools = [lire_planning_excel]

# Création du prompt pour guider l'agent
prompt_path = "prompt_syst.txt"

with open(prompt_path, 'r') as file:
    system_prompt = file.read()


#on fait un joli prompt avec notre prompt de base auquel on ajoute de la mémoire pour qu'il garde l'historique de la conversation, inupt est notre question (humaine) et le blocnotes c'est là où l'agent écrit ses pensées
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), MessagesPlaceholder(variable_name="chat_history", optional=True),("human", "{input}\n\n{agent_scratchpad}"),])

#on construit l'agent en lui donnant tout
agent = create_structured_chat_agent(llm, tools, prompt)

# on l"exécute ! 
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,  # pour voir le raisonnement
    handle_parsing_errors=True
)